In [43]:
import pandas as pd
import numpy as np

def build_24_month_cohort(my_table_file, dxsum_file):
    print("Loading data...")
    df_my = pd.read_csv(my_table_file, low_memory=False)
    df_dx = pd.read_csv(dxsum_file, low_memory=False)
    
    # df_dx VISCODE columns to visit for consistency and also subject id if needed
    if 'VISCODE' in df_dx.columns:
        df_dx.rename(columns={'VISCODE': 'visit', 'PTID': 'subject_id'}, inplace=True)

    # ---------------------------------------------------------
    # 1. Grab ALL Features from My_Table (Fixes Problem 1)
    # ---------------------------------------------------------
    # Sort to ensure 'bl' comes first, then take the first valid row per patient
    first_visits = df_my.sort_values(['subject_id', 'visit']).groupby('subject_id').first().reset_index()
    
    # Filter for baseline MCI patients
    mci_cohort = first_visits[first_visits['entry_research_group'].str.contains('MCI', na=False, case=False)].copy()
    mci_subjects = mci_cohort['subject_id'].unique()
    print(f"Identified {len(mci_subjects)} baseline MCI patients.")

    # Select the comprehensive feature list
    desired_features = [
        'subject_id', 'entry_age', 'PTGENDER', 'PTEDUCAT', 'GENOTYPE', 
        'TOTAL13', 'CDRSB', 'MMSCORE', 'FAQTOTAL', 'MOCA', 'NPISCORE'
    ]
    # Keep only what exists to prevent errors
    actual_features = [f for f in desired_features if f in mci_cohort.columns]
    mci_final_features = mci_cohort[actual_features].copy()

    # ---------------------------------------------------------
    # 2. 24-Month Window Logic (Fixes Problems 2 & 3)
    # ---------------------------------------------------------
    def get_month(v):
        v = str(v).lower().strip()
        if v in ['bl', 'sc']: return 0
        if v.startswith('m'):
            try: return int(v.replace('m', ''))
            except: return -1
        return -1
        
    df_dx['month'] = df_dx['visit'].apply(get_month)
    long_dx = df_dx[df_dx['subject_id'].isin(mci_subjects)]
    
    progressors = []
    stable = []
    
    PREDICTION_WINDOW = 24 # 24 Months (2 Years)
    
    for subj_id, group in long_dx.groupby('subject_id'):
        converted = False
        
        # Check ANY visit from Baseline up to Month 24
        window_24m = group[group['month'] <= PREDICTION_WINDOW]
        
        if 'DIAGNOSIS' in window_24m.columns and 3 in window_24m['DIAGNOSIS'].values: 
            converted = True
        if 'DXCHANGE' in window_24m.columns and any(x in [3, 5, 6] for x in window_24m['DXCHANGE'].values): 
            converted = True
            
        if converted:
            # They got AD *any time* within 24 months
            progressors.append(subj_id)
        else:
            # Did they stay in the study for AT LEAST 24 months to prove they were stable?
            if group['month'].max() >= PREDICTION_WINDOW:
                stable.append(subj_id)

    # ---------------------------------------------------------
    # 3. Apply Labels and Final Output
    # ---------------------------------------------------------
    final_df = mci_final_features[mci_final_features['subject_id'].isin(progressors + stable)].copy()
    final_df['label'] = final_df['subject_id'].apply(lambda x: 1 if x in progressors else 0)

    print("\n" + "="*50)
    print("24-MONTH MASTER DATASET READY")
    print("="*50)
    print(f"Total Valid Patients: {len(final_df)} (Progressors: {len(progressors)}, Stable: {len(stable)})")
    
    print("\nColumns in final dataset:")
    print(final_df.columns.tolist())
    
    return final_df, first_visits, long_dx

# --- RUN IT ---
my_table_path = '../data/adni/raw_data/All_Subjects_My_Table_27Mar2026.csv'
dxsum_name = '../data/adni/raw_data/All_Subjects_DXSUM_27Mar2026.csv'
final_df, first_visits, long_dx = build_24_month_cohort(my_table_path, dxsum_name)

Loading data...
Identified 1674 baseline MCI patients.

24-MONTH MASTER DATASET READY
Total Valid Patients: 543 (Progressors: 400, Stable: 143)

Columns in final dataset:
['subject_id', 'entry_age', 'PTGENDER', 'PTEDUCAT', 'GENOTYPE', 'TOTAL13', 'CDRSB', 'MMSCORE', 'FAQTOTAL', 'MOCA', 'NPISCORE', 'label']


In [44]:
# prevalence on this cohort (only bl visit to 24 months visits)
print("\nPrevalence of AD Progression within 24 Months:")
print(final_df['label'].value_counts(normalize=True))


Prevalence of AD Progression within 24 Months:
label
1    0.736648
0    0.263352
Name: proportion, dtype: float64


In [45]:
final_df

,subject_id,entry_age,PTGENDER,PTEDUCAT,GENOTYPE,TOTAL13,CDRSB,MMSCORE,FAQTOTAL,MOCA,NPISCORE,label
10,002_S_0729,65.18,2.0,16.0,3/4,14.67,0.5,27.0,7.0,NaN,0.0,1
12,002_S_0782,81.65,1.0,16.0,3/3,23.33,1.0,30.0,0.0,NaN,1.0,0
17,002_S_0954,69.40,2.0,14.0,3/4,21.67,1.5,24.0,1.0,NaN,0.0,1
25,002_S_1070,73.71,1.0,14.0,3/3,21.67,3.5,22.0,8.0,NaN,7.0,1
27,002_S_1155,57.92,1.0,20.0,3/3,17.00,1.5,27.0,1.0,25.0,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4880,941_S_1311,69.14,1.0,12.0,4/4,18.33,3.5,27.0,8.0,NaN,3.0,1
4881,941_S_1363,69.82,2.0,12.0,3/4,27.33,4.0,20.0,7.0,NaN,3.0,1
4906,941_S_4420,81.44,1.0,18.0,3/3,17.33,1.0,29.0,4.0,NaN,3.0,1
4918,941_S_6068,75.79,1.0,12.0,3/4,14.33,4.0,25.0,12.0,19.0,NaN,1


In [46]:

def build_rolling_window_cohort(date_suffix="27Mar2026", window_months=24, fuzzy_tolerance=3):
    print(f"Building Rolling-Window Cohort ({window_months}-mo horizon, {fuzzy_tolerance}-mo tolerance)...")
    
    # --- [Data Loading & Renaming as before] ---
    df_my = pd.read_csv(f'../data/adni/raw_data/All_Subjects_My_Table_{date_suffix}.csv', low_memory=False)
    df_dx = pd.read_csv(f'../data/adni/raw_data/All_Subjects_DXSUM_{date_suffix}.csv', low_memory=False)
    df_demo = pd.read_csv(f'../data/adni/raw_data/All_Subjects_PTDEMOG_{date_suffix}.csv', low_memory=False)
    df_adas = pd.read_csv(f'../data/adni/raw_data/All_Subjects_ADAS_{date_suffix}.csv', low_memory=False)
    
    for df in [df_dx, df_demo, df_adas]:
        df.rename(columns={'VISCODE': 'visit', 'PTID': 'subject_id'}, inplace=True)

    def get_month(v):
        v = str(v).lower().strip()
        if v in ['bl', 'sc']: return 0
        if v.startswith('m'):
            try: return int(v.replace('m', ''))
            except: return -1
        return -1

    df_my['month'] = df_my['visit'].apply(get_month)
    df_dx['month'] = df_dx['visit'].apply(get_month)
    df_adas['month'] = df_adas['visit'].apply(get_month)
    
    # 2. Get Static Demographics from PTDEMOG (Gender, Education)
    print("Extracting static demographics from PTDEMOG...")
    bl_demo = df_demo.sort_values(['subject_id', 'visit']).groupby('subject_id').first().reset_index()
    demo_cols = ['subject_id', 'PTGENDER', 'PTEDUCAT']
    bl_demo = bl_demo[[c for c in demo_cols if c in bl_demo.columns]]

    # 3. Fuse Longitudinal Features (including Age and APOE from My_Table)
    print("Fusing longitudinal features and genetic data...")
    
    # Identify which columns are actually in My_Table
    # We look for Age (entry_age) and APOE4 here
    my_cols = ['subject_id', 'visit', 'month', 'entry_research_group', 'entry_age', 'MMSCORE', 'CDRSB', 'FAQTOTAL', 'MOCA', 'NPISCORE', 'GENOTYPE']

    
    long_features = df_my[my_cols].copy()

    long_features['age_at_visit'] = long_features['entry_age'] + (long_features['month'] / 12)

    
    long_adas = df_adas[['subject_id', 'month', 'TOTAL13']].drop_duplicates(subset=['subject_id', 'month'])
    long_features = long_features.merge(long_adas, on=['subject_id', 'month'], how='left')
    


    # Merge Gender and Education from the PTDEMOG subset we made
    long_features = long_features.merge(bl_demo, on='subject_id', how='left')

    # ---------------------------------------------------------
    # CONSOLIDATE SC/BL (MONTH 0)
    # ---------------------------------------------------------
    print("Consolidating 'sc' and 'bl' visits to maximize Baseline features...")
    
    # Define clinical columns that might be split between sc and bl
    clin_cols = my_cols[4:]  # Assuming first 4 cols are subject_id, visit, month, entry_research_group
    
    # 1. Sort to ensure 'sc' (screening) is processed before 'bl' (baseline)
    long_features = long_features.sort_values(['subject_id', 'month', 'visit'])
    
    # 2. Within each patient's Month 0, fill NAs using both rows
    # (e.g., if MMSE is in 'sc' and ADAS is in 'bl', the 'bl' row will now have both)
    m0_mask = long_features['month'] == 0
    long_features.loc[m0_mask, clin_cols] = long_features[m0_mask].groupby('subject_id')[clin_cols].ffill().bfill()
    
    # 3. Drop the 'sc' row, keeping the 'bl' row which now has the combined data
    # If they have both, we keep 'last' (usually the 'bl' visit)
    long_features = long_features.drop_duplicates(subset=['subject_id', 'month'], keep='last')
    
    # ---------------------------------------------------------
    # --- [Label Generation with Diagnosis Gatekeeper as before] ---
    valid_rows = []
    dx_grouped = dict(tuple(df_dx.groupby('subject_id')))
    
    for idx, row in long_features.iterrows():
        subj = row['subject_id']
        current_month = row['month']
        if current_month < 0 or subj not in dx_grouped: continue
        
        patient_dx = dx_grouped[subj]
        
        # ---------------------------------------------------------
        # DYNAMIC GATEKEEPER: Strict (0) or Fuzzy (>0)
        # ---------------------------------------------------------
        if fuzzy_tolerance == 0:
            # Strict mode: Exact match only
            current_dx_row = patient_dx[patient_dx['month'] == current_month]
        else:
            # Fuzzy mode: Look for nearest record within the tolerance window
            window = patient_dx[
                (patient_dx['month'] >= current_month - fuzzy_tolerance) & 
                (patient_dx['month'] <= current_month + fuzzy_tolerance)
            ].copy()
            
            if not window.empty:
                window['dist'] = (window['month'] - current_month).abs()
                current_dx_row = window.sort_values('dist').head(1)
            else:
                current_dx_row = pd.DataFrame() # Empty if nothing in window

        # --- Check if they are MCI at this point ---
        is_mci_now = False
        if not current_dx_row.empty:
            # We use .iloc[0] because fuzzy matching returns a single-row DF
            target_row = current_dx_row.iloc[0]
            if 'DXCHANGE' in target_row.index:
                if target_row['DXCHANGE'] in [2, 4, 8]: is_mci_now = True
            elif 'DIAGNOSIS' in target_row.index:
                if target_row['DIAGNOSIS'] == 2: is_mci_now = True
        
        if not is_mci_now: continue
            
        # ---------------------------------------------------------
        # LABELING (Window-based)
        # ---------------------------------------------------------
        target_month = current_month + window_months
        future_window = patient_dx[(patient_dx['month'] > current_month) & (patient_dx['month'] <= target_month)]
        
        converted = False
        if 'DIAGNOSIS' in future_window.columns and 3 in future_window['DIAGNOSIS'].values: 
            converted = True
        if 'DXCHANGE' in future_window.columns and any(x in [3, 5, 6] for x in future_window['DXCHANGE'].values): 
            converted = True
            
        if converted:
            row['label'] = 1
            valid_rows.append(row)
        else:
            # Stable: Must have stayed in study for at least (Window - Tolerance)
            if patient_dx['month'].max() >= (target_month - fuzzy_tolerance):
                row['label'] = 0
                valid_rows.append(row)

    final_df = pd.DataFrame(valid_rows)
    def encode_apoe(genotype):
        genotype = str(genotype)
        if '4/4' in genotype: return 2
        if '4' in genotype: return 1
        return 0

    final_df['APOE4_count'] = final_df['GENOTYPE'].apply(encode_apoe)
    # Drop the original string column
    final_df.drop(columns=['GENOTYPE'], inplace=True, errors='ignore')

    
    
    # ---------------------------------------------------------
    # NEW: VISIT COUNT AUDIT
    # ---------------------------------------------------------
    print("\n" + "="*50)
    print("LONGITUDINAL VISIT AUDIT")
    print("="*50)
    
    visit_counts = final_df['subject_id'].value_counts()
    
    print(f"Total Training Rows:      {len(final_df)}")
    print(f"Unique Patients:          {len(visit_counts)}")
    print(f"Average Rows per Patient: {visit_counts.mean():.2f}")
    print(f"Max Rows from one Patient: {visit_counts.max()}")
    print(f"Min Rows from one Patient: {visit_counts.min()}")
    
    print("\n--- Rows per Patient Distribution ---")
    dist = visit_counts.value_counts().sort_index()
    print(f"{'Rows':<10} | {'Number of Patients'}")
    print("-" * 30)
    for num_rows, num_patients in dist.items():
        print(f"{num_rows:<10} | {num_patients}")
        
    print("\n" + "="*50)
    print(f"Prior (Prevalence): {round(final_df['label'].mean(), 3)}")
    print("="*50)
    
    return final_df

rolling_dataset = build_rolling_window_cohort(date_suffix="27Mar2026", window_months=24)

Building Rolling-Window Cohort (24-mo horizon, 3-mo tolerance)...
Extracting static demographics from PTDEMOG...
Fusing longitudinal features and genetic data...
Consolidating 'sc' and 'bl' visits to maximize Baseline features...

LONGITUDINAL VISIT AUDIT
Total Training Rows:      1129
Unique Patients:          343
Average Rows per Patient: 3.29
Max Rows from one Patient: 7
Min Rows from one Patient: 1

--- Rows per Patient Distribution ---
Rows       | Number of Patients
------------------------------
1          | 52
2          | 53
3          | 105
4          | 40
5          | 63
6          | 29
7          | 1

Prior (Prevalence): 0.43


APPLY LOCF

In [47]:
def prep_and_save_data(rolling_df):
    df = rolling_df.copy()
    
    print("Applying patient-specific forward-fill (LOCF)...")
    # Sort chronologically so forward-fill works correctly
    df = df.sort_values(['subject_id', 'month'])
    
    # Safely impute longitudinally 
    clinical_scores = ['TOTAL13', 'MMSCORE', 'CDRSB', 'FAQTOTAL', 'MOCA']
    df[clinical_scores] = df.groupby('subject_id')[clinical_scores].ffill()
    
    # Format Gender
    if df['PTGENDER'].dtype == 'O':
        df['PTGENDER'] = df['PTGENDER'].map({'Male': 0, 'Female': 1, 'M': 0, 'F': 1})
        df['PTGENDER'] = pd.to_numeric(df['PTGENDER'], errors='coerce').fillna(0)
  
    
    return df

# --- RUN IT ---
final_df = prep_and_save_data(rolling_dataset)

Applying patient-specific forward-fill (LOCF)...


In [48]:
cols_to_drop = [
    'month',                # Use age_at_visit instead to avoid study-duration leakage
    'visit',                # Metadata only
    'entry_age',            # Redundant with age_at_visit
    'AGE',                  # Redundant with age_at_visit
    'entry_research_group'  # We already filtered for MCI; this is now administrative noise
]

final_features_df = final_df.drop(columns=[c for c in cols_to_drop if c in rolling_dataset.columns])

print(f"Post-cleanup columns: {final_features_df.columns.tolist()}")

Post-cleanup columns: ['subject_id', 'MMSCORE', 'CDRSB', 'FAQTOTAL', 'MOCA', 'NPISCORE', 'age_at_visit', 'TOTAL13', 'PTGENDER', 'PTEDUCAT', 'label', 'APOE4_count']


In [49]:
final_features_df[['subject_id', 'MMSCORE', 'CDRSB', 'FAQTOTAL', 'MOCA', 
                   'NPISCORE', 'age_at_visit', 'TOTAL13', 'PTGENDER', 'PTEDUCAT', 
                   'APOE4_count', 'label']].to_csv("../data/adni/adni_rolling_locf.csv", index=False)

In [ ]:
final_df

,subject_id,visit,month,entry_research_group,MMSCORE,CDRSB,FAQTOTAL,TOTAL13,PTGENDER,PTEDUCAT,label
10593,002_S_0729,sc,0,MCI,27.0,0.5,7.0,14.67,2.0,16.0,1
10577,002_S_0729,m06,6,MCI,27.0,0.5,5.0,17.00,2.0,16.0,1
10900,002_S_0782,sc,0,MCI,29.0,0.5,0.0,23.33,1.0,16.0,0
10893,002_S_0782,m06,6,MCI,30.0,1.0,2.0,16.00,1.0,16.0,0
10894,002_S_0782,m12,12,MCI,28.0,1.0,0.0,12.33,1.0,16.0,0
...,...,...,...,...,...,...,...,...,...,...,...
14075,941_S_1295,m06,6,MCI,25.0,1.5,1.0,20.00,1.0,16.0,1
14076,941_S_1295,m12,12,MCI,24.0,2.5,1.0,26.00,1.0,16.0,1
14288,941_S_1311,sc,0,MCI,29.0,2.5,8.0,18.33,1.0,12.0,1
14284,941_S_1311,m06,6,MCI,27.0,3.5,16.0,17.00,1.0,12.0,1
